# Transformer sliding-window churn (future 10 days)

滑动窗口示例：以每天为锚点，标签为后续 10 天是否出现 `Cancellation Confirmation`。特征沿用现有 `feature_pipeline`，不修改原代码。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import log_loss
import torch
from torch.utils.data import DataLoader

from feature_pipeline import feature_engineer, encode_categoricals, _build_mapping, _infer_numeric_cols
from transformer_model import (
    TrainingConfig,
    ChurnTransformer,
    SequenceData,
    UserSequenceDataset,
    collate_batch,
    train_model,
    predict_proba,
)

PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "churn-prediction-25-26"
TRAIN_PATH = DATA_DIR / "train.parquet"
TEST_PATH = DATA_DIR / "test.parquet"
OUTPUT_DIR = Path("outputs_transformer_sliding")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HORIZON_DAYS = 10
ANCHOR_END = pd.Timestamp("2018-11-09")  # 最后一个锚点日（含），None 则用数据末尾-horizon
MAX_SEQ_LEN = 400
VAL_FRAC = 0.2

CONFIG = TrainingConfig(max_seq_len=MAX_SEQ_LEN, batch_size=32, epochs=6, lr=1e-3)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
def prepare_event_features(train_path: Path, test_path: Path):
    train_raw = pd.read_parquet(train_path)
    test_raw = pd.read_parquet(test_path)

    # 留下原始含 churn 的副本用于标签，特征构建阶段去掉 churn 行防止泄露
    label_raw = train_raw.copy()
    train_no_churn = train_raw[train_raw["page"] != "Cancellation Confirmation"].copy()
    test_no_churn = test_raw.copy()

    page_categories = sorted(set(train_no_churn["page"]).union(set(test_no_churn["page"])))
    train_fe = feature_engineer(train_no_churn, page_categories)
    test_fe = feature_engineer(test_no_churn, page_categories)

    metro_map = _build_mapping([train_fe["metro"], test_fe["metro"]])
    state_map = _build_mapping([train_fe["state"], test_fe["state"]])
    device_map = _build_mapping([train_fe["device"], test_fe["device"]])

    train_fe = encode_categoricals(train_fe, page_categories, metro_map, state_map, device_map)
    test_fe = encode_categoricals(test_fe, page_categories, metro_map, state_map, device_map)

    for df in (train_fe, test_fe):
        df["prev_page_id"] = df.groupby("userId")["page_id"].shift(1).fillna(0).astype(np.int64)
        if "error occur" in df.columns:
            df.rename(columns={"error occur": "error_occur"}, inplace=True)
        df["regis_time_seconds"] = df["regis_time"].dt.total_seconds()

    drop_cols = {"regis_time", "song", "artist", "page", "metro", "state", "device"}
    train_fe = train_fe.drop(columns=[c for c in drop_cols if c in train_fe.columns])
    test_fe = test_fe.drop(columns=[c for c in drop_cols if c in test_fe.columns])

    numeric_cols = sorted(set(_infer_numeric_cols(train_fe) + ["regis_time_seconds"]))
    for df in (train_fe, test_fe):
        for col in numeric_cols:
            if col not in df:
                df[col] = 0.0
        df[numeric_cols] = df[numeric_cols].fillna(0)

    scaler = StandardScaler()
    train_fe[numeric_cols] = scaler.fit_transform(train_fe[numeric_cols])
    test_fe[numeric_cols] = scaler.transform(test_fe[numeric_cols])

    cat_sizes = {
        "pages": len(page_categories),
        "metro": len(metro_map),
        "state": len(state_map),
        "device": len(device_map),
    }

    return train_fe, test_fe, label_raw, numeric_cols, cat_sizes


def build_sliding_samples(feat_df: pd.DataFrame, label_raw: pd.DataFrame, numeric_cols, horizon_days: int, anchor_end: pd.Timestamp | None, max_seq_len: int):
    horizon = pd.Timedelta(days=horizon_days)
    churn_times = (
        label_raw[label_raw["page"] == "Cancellation Confirmation"]
        .groupby("userId")["time"]
        .apply(list)
        .to_dict()
    )

    sequences = {}
    labels = {}
    anchors = {}

    grouped = feat_df.sort_values(["userId", "time"]).groupby("userId", sort=False)
    for uid, g in grouped:
        g = g.reset_index(drop=True)
        if g.empty:
            continue
        last_valid = (g["time"].max() - horizon).normalize()
        days = pd.to_datetime(g["time"].dt.normalize().unique())
        for day in days:
            if day > last_valid:
                continue
            if anchor_end is not None and day > anchor_end:
                continue
            cutoff = day + pd.Timedelta(days=1)
            window = g[g["time"] < cutoff]
            if window.empty:
                continue
            sample_id = f"{uid}__{day.date()}"
            anchors[sample_id] = day

            label = 0
            for t in churn_times.get(uid, []):
                if day < t <= day + horizon:
                    label = 1
                    break
            labels[sample_id] = label

            seq = window.tail(max_seq_len)
            sequences[sample_id] = SequenceData(
                numeric=seq[numeric_cols].to_numpy(np.float32),
                page_ids=seq["page_id"].to_numpy(np.int64),
                prev_page_ids=seq["prev_page_id"].to_numpy(np.int64),
                metro_ids=seq["metro_id"].to_numpy(np.int64),
                state_ids=seq["state_id"].to_numpy(np.int64),
                device_ids=seq["device_id"].to_numpy(np.int64),
            )

    return sequences, labels, anchors


def build_test_sequences(feat_df: pd.DataFrame, numeric_cols, horizon_days: int, anchor_end: pd.Timestamp | None, max_seq_len: int):
    horizon = pd.Timedelta(days=horizon_days)
    sequences = {}
    anchors = {}
    grouped = feat_df.sort_values(["userId", "time"]).groupby("userId", sort=False)
    for uid, g in grouped:
        if g.empty:
            continue
        candidate = (g["time"].max() - horizon).normalize()
        anchor_day = candidate if anchor_end is None else min(candidate, anchor_end)
        cutoff = anchor_day + pd.Timedelta(days=1)
        window = g[g["time"] < cutoff]
        if window.empty:
            window = g
        sample_id = f"{uid}__test"
        anchors[sample_id] = {"userId": uid, "anchor_date": anchor_day}
        seq = window.tail(max_seq_len)
        sequences[sample_id] = SequenceData(
            numeric=seq[numeric_cols].to_numpy(np.float32),
            page_ids=seq["page_id"].to_numpy(np.int64),
            prev_page_ids=seq["prev_page_id"].to_numpy(np.int64),
            metro_ids=seq["metro_id"].to_numpy(np.int64),
            state_ids=seq["state_id"].to_numpy(np.int64),
            device_ids=seq["device_id"].to_numpy(np.int64),
        )
    return sequences, anchors


def time_based_split(anchors: dict, val_frac: float = 0.2):
    ids = list(anchors.keys())
    anchor_dates = pd.to_datetime([anchors[i] for i in ids])
    split_date = anchor_dates.quantile(1 - val_frac)
    train_ids = [i for i, d in zip(ids, anchor_dates) if d <= split_date]
    val_ids = [i for i in ids if i not in train_ids]
    return train_ids, val_ids


def make_loader(sample_ids, sequences, labels_dict, batch_size, shuffle):
    ds = UserSequenceDataset(
        sequences={sid: sequences[sid] for sid in sample_ids},
        labels=labels_dict,
        user_ids=sample_ids,
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0, collate_fn=collate_batch)


train_fe, test_fe, label_raw, numeric_cols, cat_sizes = prepare_event_features(TRAIN_PATH, TEST_PATH)
sequences, labels, anchors = build_sliding_samples(train_fe, label_raw, numeric_cols, HORIZON_DAYS, ANCHOR_END, MAX_SEQ_LEN)
train_ids, val_ids = time_based_split(anchors, VAL_FRAC)

print(f"Train samples: {len(train_ids)}, Val samples: {len(val_ids)}, Pos rate: {np.mean(list(labels.values())):.3f}")

In [ ]:
train_loader = make_loader(train_ids, sequences, labels, CONFIG.batch_size, True)
val_loader = make_loader(val_ids, sequences, labels, CONFIG.batch_size, False)

pos = sum(labels[sid] for sid in train_ids)
neg = len(train_ids) - pos
pos_weight = neg / max(pos, 1)

model = ChurnTransformer(
    num_numeric=len(numeric_cols),
    num_pages=cat_sizes["pages"],
    num_metro=cat_sizes["metro"],
    num_state=cat_sizes["state"],
    num_device=cat_sizes["device"],
    config=CONFIG,
).to(DEVICE)

train_history, val_history, model, best_epoch, best_state = train_model(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    device=DEVICE,
    epochs=CONFIG.epochs,
    lr=CONFIG.lr,
    weight_decay=CONFIG.weight_decay,
    pos_weight=pos_weight,
    use_cosine_decay=CONFIG.use_cosine_decay,
    eta_min_factor=CONFIG.eta_min_factor,
    warmup_epochs=CONFIG.warmup_epochs,
    use_focal_loss=CONFIG.use_focal_loss,
    focal_gamma=CONFIG.focal_gamma,
    early_stop_patience=CONFIG.early_stop_patience,
    early_stop_min_delta=CONFIG.early_stop_min_delta,
)

val_probs = predict_proba(model, val_loader, DEVICE)
val_labels = pd.Series({sid: labels[sid] for sid in val_probs.keys()})
val_pred = pd.Series(val_probs)
val_logloss = log_loss(val_labels, val_pred)
print(f"Best epoch: {best_epoch}, Val logloss: {val_logloss:.4f}")

torch.save(best_state, OUTPUT_DIR / "transformer_sliding.pt")

In [ ]:
test_sequences, test_meta = build_test_sequences(test_fe, numeric_cols, HORIZON_DAYS, ANCHOR_END, MAX_SEQ_LEN)
test_loader = DataLoader(
    UserSequenceDataset(test_sequences, labels=None, user_ids=list(test_sequences.keys())),
    batch_size=CONFIG.batch_size,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_batch,
)

test_probs = predict_proba(model, test_loader, DEVICE)
rows = []
for sid, prob in test_probs.items():
    meta = test_meta[sid]
    rows.append({"id": meta["userId"], "target": int(prob >= 0.5), "probability": prob})
submission = pd.DataFrame(rows)
submission_path = OUTPUT_DIR / "transformer_sliding_submission.csv"
submission.to_csv(submission_path, index=False)
submission.head()